# client-stdio: Talking to the server over stdio

**Over stdio, the client starts the server itself, as a child process, and the two talk through the server's stdin and stdout.** You don't start anything in a terminal first.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 this notebook (client)          python server.py stdio (child)
 ----------------------          ------------------------------
 writes a JSON message  --stdin--&gt;  reads it, runs the request
 reads the reply        &lt;-stdout--  writes one JSON line back
                                    logs --stderr--&gt; server_log.txt
</pre>

Read `server.ipynb` first. Section 6 of `../1-understanding-mcp/README.md` shows the raw JSON messages this notebook sends.

## 1. The basics: A child process and its streams

- **Child process:** A program started by another program. Here the client runs `python server.py stdio`, just as you would type it in a terminal.
- **stdin and stdout:** Every program gets an input stream (stdin) and an output stream (stdout). `input()` reads from stdin, and `print()` writes to stdout. The program that starts a child holds the other ends of the child's streams, so it can write into the child's stdin and read from its stdout.
- **stderr:** A third stream, for logs and errors. MCP keeps it out of the conversation.

That's all stdio is: The JSON-RPC messages, one per line, travelling through these streams.

## 2. Saying how to start the server

`StdioServerParameters` is the SDK's class for "the command that starts the server". `command` is the program and `args` are the words after it, so this describes `python server.py stdio`. `stdio_client` and `ClientSession` are imported here and explained in section 3.

In [1]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command="python", args=["server.py", "stdio"])
print(server)

command='python' args=['server.py', 'stdio'] env=None cwd=None encoding='utf-8' encoding_error_handler='strict'


`env=None` and `cwd=None` mean the server gets the same environment variables and the same folder as this notebook, which is how `server.py` is found.

## 3. Connecting, listing the tools and calling `add`

Three layers do the work, one inside the other:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 session.call_tool("add", ...)   ClientSession: Writes the JSON-RPC
          |                      request and waits for its reply
 read and write streams          stdio_client: Pipes to the child
          |
 python server.py stdio          The server: Runs add(2, 3)
</pre>

- **`stdio_client(server)`:** Starts the child process and gives back two streams: `read` for messages from the server, and `write` for messages to it.
- **`ClientSession(read, write)`:** Speaks MCP over those two streams. `initialize()` does the handshake, `list_tools()` sends `tools/list`, and `call_tool(...)` sends `tools/call`.
- **`async with`:** Opens each layer and closes it when its block ends. Closing them also ends the server process.
- **`await`:** Waits for the server's reply. A notebook lets you use it directly in a cell.
- **`errlog=log`:** In a terminal, the server's stderr shows on your screen. A notebook on Windows has no real screen stream to give the child process, so we give it a file, `server_log.txt`, and read it in section 5. `with open("server_log.txt", "w") as log:` opens the file for writing and closes it at the end.

In [2]:
with open("server_log.txt", "w") as log:
    async with stdio_client(server, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            init_result = await session.initialize()
            tools_result = await session.list_tools()
            result = await session.call_tool("add", {"a": 2, "b": 3})

print(result.content[0].text)

5


The three results stay in `init_result`, `tools_result` and `result` after the connection closes, so the next cells can look at them one at a time.

## 4. What came back

#### 4.1 The handshake reply

`initialize()` returns an `InitializeResult`: The server's answer to `initialize`.

In [3]:
print(type(init_result))
print(init_result.model_dump_json(indent=2, exclude_none=True))

<class 'mcp.types.InitializeResult'>
{
  "protocolVersion": "2025-11-25",
  "capabilities": {
    "experimental": {},
    "prompts": {
      "listChanged": false
    },
    "resources": {
      "subscribe": false,
      "listChanged": false
    },
    "tools": {
      "listChanged": false
    }
  },
  "serverInfo": {
    "name": "Calculator",
    "version": "1.27.0"
  }
}


- **`protocolVersion`:** The spec version both sides agreed on.
- **`capabilities`:** What the server offers. FastMCP lists tools, resources and prompts even though this server has only a tool.
- **`serverInfo`:** The name from `FastMCP("Calculator", ...)`, with the SDK's version.

#### 4.2 The tool list

`list_tools()` returns a `ListToolsResult`. Its `.tools` field holds the list of tools, which is why the code reads `tools_result.tools[0]`.

In [4]:
print(type(tools_result))
print(len(tools_result.tools))
print(tools_result.tools[0].name)

<class 'mcp.types.ListToolsResult'>
1
add


Each tool carries the name, description and schemas you saw in `server.ipynb`, section 2. A host, like the one in Part 3, hands these to the model.

#### 4.3 The tool result

`call_tool(...)` returns a `CallToolResult`.

In [5]:
print(type(result))
print(result.model_dump_json(indent=2, exclude_none=True))

<class 'mcp.types.CallToolResult'>
{
  "content": [
    {
      "type": "text",
      "text": "5"
    }
  ],
  "structuredContent": {
    "result": 5
  },
  "isError": false
}


- **`content`:** The answer as a list of content blocks. Here it's one text block, `"5"`. This is what goes back to the model.
- **`structuredContent`:** The same answer as typed JSON, `{"result": 5}`, matching the tool's output schema.
- **`isError`:** `false`, so the tool ran without failing.

In [6]:
print(type(result.content[0].text))
print(result.structuredContent["result"])

<class 'str'>
5


The answer in `content` travels as the text `"5"`, not the number 5. Code that needs the number reads it from `structuredContent`.

## 5. What the server logged

The file holds everything the server wrote to stderr:

In [7]:
with open("server_log.txt") as log:
    print(log.read())

Calculator server starting with transport: stdio
Processing request of type ListToolsRequest
Processing request of type CallToolRequest



The startup line and FastMCP's "Processing request" lines all went to stderr. None of them mixed with the JSON on stdout, which is why `server.py` prints to stderr.

## 6. The .py version

`client-stdio.py` does the same in a plain script. Two things differ:

- **`async def main()` and `asyncio.run(main())`:** A plain Python file can't use `await` at the top level, so the code goes inside an async function, and `asyncio.run` runs it.
- **No `errlog`:** In a terminal, the server's stderr goes straight to your screen.

<div style="font-size: 0.85em">

```python
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command="python", args=["server.py", "stdio"])


async def main():
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            tool = tools_result.tools[0]
            print(tool.name)
            print(tool.description)

            result = await session.call_tool("add", {"a": 2, "b": 3})
            print(result.content[0].text)


asyncio.run(main())
```

</div>

Run it from this folder with `python client-stdio.py`:

<div style="font-size: 0.85em">

```text
Calculator server starting with transport: stdio
Processing request of type ListToolsRequest
add
Add two numbers together
Processing request of type CallToolRequest
5
```

</div>

The server's stderr lines and the client's prints share one screen, so they interleave.

## Say it out loud

> "Over stdio, my client starts the server as a child process, described by `StdioServerParameters`, and they exchange JSON-RPC messages, one per line, over the server's stdin and stdout. `stdio_client` gives me the read and write streams, and `ClientSession` speaks MCP over them: `initialize`, `list_tools`, `call_tool`. The server logs to stderr, because stdout is the protocol channel. It's the right transport for a local server with one client, like a desktop app or an IDE starting a tool."